In [73]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np
import zipfile
from pathlib import Path
import shutil
import time
import gc
import tempfile

In [74]:
data_path = Path(r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\Earthquakes")

In [77]:
# Loop through each ZIP file in the data path
for zip_file_path in data_path.glob('*.zip'):
    # Define the output raster file path based on the ZIP file name
    output_raster_path = data_path / f"{zip_file_path.stem}.tif"

    # Create a unique temporary directory for this iteration
    with tempfile.TemporaryDirectory() as temp_dir_str:
        temp_dir = Path(temp_dir_str)

        # Extract the shapefile and associated files from the ZIP file
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # Extract all files to the temporary directory
            zip_ref.extractall(path=temp_dir)

            # Navigate to the 'output' folder within the extracted files
            output_folder = temp_dir / 'output'
            if not output_folder.exists():
                print(f"'output' folder not found in {zip_file_path}")
                continue

            # Find the shapefile name within the 'output' folder
            shapefile_name = next((file for file in output_folder.iterdir() if file.suffix == '.shp'), None)
            if shapefile_name is None:
                print(f"No shapefile found in 'output' folder of {zip_file_path}")
                continue

            # Load the shapefile
            gdf = gpd.read_file(shapefile_name).iloc[1:]

            # Define the resolution and extent of the raster
            resolution = (gdf.iloc[1].geometry.area / 2, gdf.iloc[1].geometry.area / 2)  # Example resolution

            # Calculate the extent of the shapefile
            minx, miny, maxx, maxy = gdf.total_bounds

            # Calculate the width and height of the raster
            width = int((maxx - minx) / resolution[0])
            height = int((maxy - miny) / resolution[1])

            # Create a transform object
            transform = rasterio.transform.from_origin(minx, maxy, resolution[0], resolution[1])

            # Rasterize the shapefile with HPVALUE
            shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf['HPVALUE']))
            rasterized = rasterize(
                shapes,
                out_shape=(height, width),
                fill=0,
                transform=transform,
                all_touched=False,
                dtype=np.float32
            )

            # Check if rasterization produced any non-zero values
            print(f"Unique values in rasterized array for {zip_file_path.stem}:", len(np.unique(rasterized)))

            # Write the raster to a file
            with rasterio.open(
                output_raster_path, 'w',
                driver='GTiff',
                height=height,
                width=width,
                count=1,
                dtype=rasterized.dtype,
                crs=gdf.crs,
                compress = 'LZW',
                transform=transform
            ) as dst:
                dst.write(rasterized, 1)

            print(f"Raster file saved to {output_raster_path}")

        # Force garbage collection to clean up any remaining file handles
        gc.collect()

Unique values in rasterized array for PGA_1_101_vs30: 82048
Raster file saved to C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\Earthquakes\PGA_1_101_vs30.tif
Unique values in rasterized array for PGA_1_2500_vs30: 100483
Raster file saved to C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\Earthquakes\PGA_1_2500_vs30.tif
Unique values in rasterized array for PGA_1_476_vs30: 98410
Raster file saved to C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\Earthquakes\PGA_1_476_vs30.tif
Unique values in rasterized array for PGA_1_5000_vs30: 100492
Raster file saved to C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\Earthquakes\PGA_1_5000_vs30.tif
Unique values in rasterized array for PGA_1_50_vs30: 62798
Raster file saved to C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP